In [ ]:
# declare libraries
import itertools
import json
import os
import pathlib
import pprint as pp
import sys
import time
from zipfile import ZipFile

import geojson
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import requests
import rioxarray as rxr
import shapely
import xarray as xr
from planet import Auth, DataClient, OrdersClient, Planet, Session
from rasterio.plot import show
from requests.auth import HTTPBasicAuth
from shapely.geometry import mapping, shape
from shapely.ops import unary_union
from shapely.validation import explain_validity

sys.path.append("../utils")

import aoi_filter_maker as aoi
import config
import pyproj
from pyproj import CRS

# bring in 2019 buffer geometries geojson for testing
# buffer_geometries_2019.geojson
for yr in range(2019, 2024):
    with open(
        f"/capstone/wildfire_prep/data/training_geometries/training_geometries_{yr}.geojson",
        "r",
    ) as file:
        globals()[f"training_{yr}"] = dict(geojson.load(file))["features"]


buffer_2019_df = gpd.read_file(
    "/capstone/wildfire_prep/data/training_geometries/training_geometries_2019.geojson"
)

# declare misc vars
crs = "EPSG:4326"
# crs = "EPSG:3310"
county_filepath = "/capstone/wildfire_prep/data/ca_counties/CA_Counties.shp"
# sb_bbox = [-125, 34.25, -119.0, 38.0]

data_api_url = "https://api.planet.com/data/v1"
orders_api_url = "https://api.planet.com/compute/ops/orders/v2"

In [2]:
# t_before = time.time()
# print("test str")
# t_after = time.time()

# print(f"Total execution time: {t_after - t_before} ")

In [3]:
# pp.pprint(
#     training_2019
#     )

Set api key and authorize

In [ ]:
planet_key = config.planet_api

# authentication
auth = HTTPBasicAuth(planet_key, "")

# check we are communicating with http properly
response = requests.get(data_api_url, auth=auth)
print(response)


# make function for pagenation
def p(data):
    print(json.dumps(data, indent=2))

<Response [200]>


### Init session

Enables the use of certain functions.

In [5]:
# setup
session = requests.Session()

# authenticate
session.auth = (planet_key, "")

res = session.get(data_api_url)
res

<Response [200]>

### Make AOI filters

We will make test cases for both polygon limits, and vertices limits.

In [ ]:
# make test cases for stress test
poly_1 = aoi.make_aoi_geojson(buffer_2019_df, end=1)
poly_10 = aoi.make_aoi_geojson(buffer_2019_df, end=10)
poly_25 = aoi.make_aoi_geojson(buffer_2019_df, end=25)
poly_30 = aoi.make_aoi_geojson(buffer_2019_df, end=30)
poly_40 = aoi.make_aoi_geojson(buffer_2019_df, end=40)

vert_10 = aoi.make_aoi_geojson(buffer_2019_df, max_verts=10)
vert_25 = aoi.make_aoi_geojson(buffer_2019_df, max_verts=25)
vert_60 = aoi.make_aoi_geojson(buffer_2019_df, max_verts=60)
vert_100 = aoi.make_aoi_geojson(buffer_2019_df, max_verts=100)
vert_150 = aoi.make_aoi_geojson(buffer_2019_df, max_verts=150)
vert_200 = aoi.make_aoi_geojson(buffer_2019_df, max_verts=200)
vert_300 = aoi.make_aoi_geojson(buffer_2019_df, max_verts=300)
vert_400 = aoi.make_aoi_geojson(buffer_2019_df, max_verts=400)

poly_40_june = aoi.make_aoi_geojson(buffer_2019_df, month_vals=[6], end=40)

# test_cases = [poly_1, poly_10, poly_25, poly_30, poly_40,
#               vert_10, vert_25, vert_60, vert_100, vert_150, vert_200, vert_300, vert_400]

# test_cases = [poly_25, poly_30, poly_40,
#               vert_10, vert_25, vert_60, vert_100, vert_150, vert_200, vert_300, vert_400]

# test_cases = [vert_400]


Total vertices in this AOI filter: 6
Total vertices in this AOI filter: 60
Total vertices in this AOI filter: 150
Total vertices in this AOI filter: 180
Total vertices in this AOI filter: 240
Max vert limit reached. Total vertices in this AOI filter: 6
Max vert limit reached. Total vertices in this AOI filter: 24
Max vert limit reached. Total vertices in this AOI filter: 60
Max vert limit reached. Total vertices in this AOI filter: 96
Max vert limit reached. Total vertices in this AOI filter: 150
Max vert limit reached. Total vertices in this AOI filter: 198
Max vert limit reached. Total vertices in this AOI filter: 300
Max vert limit reached. Total vertices in this AOI filter: 396
Total vertices in this AOI filter: 240


In [7]:
feb_2019_ch = aoi.make_convex_hull(buffer_2019_df, month_vals=[2])
feb_2019_ch

Vert count: 10
Hull area: 24149627.34695812


{'type': 'MultiPolygon',
 'coordinates': [[[[-119.627265, 34.438847],
    [-120.435944, 34.685423],
    [-120.435949, 34.686348],
    [-120.43485, 34.686352],
    [-119.625502, 34.442906],
    [-119.624952, 34.442589],
    [-119.624566, 34.441785],
    [-119.62457, 34.441077],
    [-119.626235, 34.438844],
    [-119.627265, 34.438847]]]]}

In [ ]:
full_2019_ch = [
    aoi.make_convex_hull(buffer_2019_df, month_vals=[i]) for i in range(1, 13)
]
full_2019_ch

Vert count: 10
Hull area: 24149627.34695812
Vert count: 10
Hull area: 385710.01110722957
Vert count: 19
Hull area: 1774754515.1649616
Vert count: 23
Hull area: 5599580589.817289
Vert count: 20
Hull area: 4996010778.277063
Vert count: 20
Hull area: 5022508542.810573
Vert count: 17
Hull area: 945075475.377007
Vert count: 28
Hull area: 668017687.3486702
Vert count: 16
Hull area: 2225789567.1880884
Vert count: 18
Hull area: 304457687.02945924


[None,
 {'type': 'MultiPolygon',
  'coordinates': [[[[-119.627265, 34.438847],
     [-120.435944, 34.685423],
     [-120.435949, 34.686348],
     [-120.43485, 34.686352],
     [-119.625502, 34.442906],
     [-119.624952, 34.442589],
     [-119.624566, 34.441785],
     [-119.62457, 34.441077],
     [-119.626235, 34.438844],
     [-119.627265, 34.438847]]]]},
 None,
 {'type': 'MultiPolygon',
  'coordinates': [[[[-119.667286, 34.4414],
     [-119.668091, 34.442105],
     [-119.668087, 34.443042],
     [-119.657791, 34.453899],
     [-119.656703, 34.453895],
     [-119.656707, 34.4531],
     [-119.664221, 34.442327],
     [-119.665365, 34.441655],
     [-119.666314, 34.441397],
     [-119.667286, 34.4414]]]]},
 {'type': 'MultiPolygon',
  'coordinates': [[[[-119.645032, 34.44085],
     [-120.014472, 34.611048],
     [-120.093922, 34.648876],
     [-120.27574, 34.746866],
     [-120.280514, 34.762348],
     [-120.280518, 34.763263],
     [-120.271158, 34.773407],
     [-120.267084, 34.775598

In [9]:
full_2019_ch[3]

{'type': 'MultiPolygon',
 'coordinates': [[[[-119.667286, 34.4414],
    [-119.668091, 34.442105],
    [-119.668087, 34.443042],
    [-119.657791, 34.453899],
    [-119.656703, 34.453895],
    [-119.656707, 34.4531],
    [-119.664221, 34.442327],
    [-119.665365, 34.441655],
    [-119.666314, 34.441397],
    [-119.667286, 34.4414]]]]}

In [10]:
# test_cases = full_2019_ch

##### Make geojson collections for testing capacity for multiple orders

In [11]:
collection_2019 = aoi.make_aoi_geojson_collection(training_2019, 40)

step: begin = 0, end = 40
step: begin = 40, end = 80
step: begin = 80, end = 120
step: begin = 120, end = 160
step: begin = 160, end = 200
step: begin = 200, end = 240
step: begin = 240, end = 280
step: begin = 280, end = 320
step: begin = 320, end = 360
step: begin = 360, end = 400
step: begin = 400, end = 440
step: begin = 440, end = 480
step: begin = 480, end = 520
step: begin = 520, end = 560
step: begin = 560, end = 600
step: begin = 600, end = 640
step: begin = 640, end = 680
step: begin = 680, end = 720
step: begin = 720, end = 760
step: begin = 760, end = 800
step: begin = 800, end = 840
step: begin = 840, end = 880
step: begin = 880, end = 920
step: begin = 920, end = 960
step: begin = 960, end = 1000
step: begin = 1000, end = 1040
step: begin = 1040, end = 1080
step: begin = 1080, end = 1120
step: begin = 1120, end = 1160
step: begin = 1160, end = 1200
step: begin = 1200, end = 1240
step: begin = 1240, end = 1280
step: begin = 1280, end = 1320
step: begin = 1320, end = 1360
s

In [12]:
len(collection_2019)

367

In [13]:
# test_cases = collection_2019[251:254]

In [14]:
# test_cases

### Declare functions
1. Making the order
2. Pinging the api for success/general order status

In [ ]:
def place_order(request, auth, retry_counter):
    # make order request
    response = requests.post(
        orders_api_url, data=json.dumps(request), auth=auth, headers=headers
    )
    print(response.json())

    print(f"Status code: {response.status_code}")
    print(f"Status comments: {response.text}")

    # if there is a rate limiting problem
    if response.status_code == 429:
        while retry_counter < 5 and response.status_code == 429:
            retry_after = int(
                response.headers.get("Retry-After", 60)
            )  # set 60 secs for the waiting time
            print(f"Rate limit hit. Retrying after {retry_after} seconds...")

            time.sleep(retry_after)  # wait for 60 secs
            retry_counter = retry_counter + 1  # increment the retry counter

            place_order(request, auth, retry_counter)  # try the function again

    elif response.status_code == 400:
        print("Bad Request, cancelling this order...")
        return None

    # get ids of scenes
    order_id = response.json()["id"]
    print(order_id)

    # construct the url of our order
    order_url = orders_api_url + "/" + order_id

    return order_url

In [ ]:
def poll_for_success(order_url, auth, num_loops=999):
    i = 0
    while i < num_loops:
        # iterate
        i += 1

        # get order request
        r = requests.get(order_url, auth=auth)
        response = r.json()

        # grab current state
        state = response["orders"][0]["state"]
        print(state)

        # compare it to a variety of end states and print it
        end_states = ["success", "failed", "partial"]
        if state in end_states:
            print(f"End State: {state}")
            break

        # wait 30 secs
        time.sleep(30)


In [17]:
# get variable name as string
def var_name_as_str(var):
    for name, value in globals().items():
        if value is var:
            return name


In [19]:
# for case in test_cases:
#     if case is None:
#         continue
#     print(case)

### run whole stress test

* capturing...
    1. how long each test case takes
    2. if each test case is successful

In [31]:
[*["2019"] * 12, "2020"]

['2019',
 '2019',
 '2019',
 '2019',
 '2019',
 '2019',
 '2019',
 '2019',
 '2019',
 '2019',
 '2019',
 '2019',
 '2020']

In [ ]:
month_nums = [
    "01",
    "02",
    "03",
    "04",
    "05",
    "06",
    "07",
    "08",
    "09",
    "10",
    "11",
    "12",
    "01",
]
year_iter = [*["2019"] * 12, "2020"]
year_nums = ["2019", "2020", "2021", "2022", "2023"]
lte_formats = [
    "31T23:59:59.999Z",  # jan
    "28T23:59:59.999Z",  # feb
    "31T23:59:59.999Z",  # mar
    "30T23:59:59.999Z",  # apr
    "31T23:59:59.999Z",  # may
    "30T23:59:59.999Z",  # jun
    "31T23:59:59.999Z",  # july
    "31T23:59:59.999Z",  # aug
    "30T23:59:59.999Z",  # sep
    "31T23:59:59.999Z",  # oct
    "30T23:59:59.999Z",  # nov
    "31T23:59:59.999Z",  # dec
]

# for num, end_format in zip(month_nums, lte_formats):
#     print(f"2019-{num}-{end_format}")

# for year, num in zip(year_iter, month_nums):
#     print(f"{year}-{num}-01T00:00:00.000Z")

len(year_iter) == len(month_nums)

for i in range(len(year_iter) - 1):
    print(f"before: {year_iter[i]}-{month_nums[i]}-01T00:00:00.000Z")
    print(f"after: {year_iter[i + 1]}-{month_nums[i + 1]}-01T00:00:00.000Z\n")

# for i in range(len(month_nums) - 1):
#     year = 2019
#     if i == len(month_nums) - 1:
#         year = 2020

#     print(f"{month_nums[i]} | {month_nums[i + 1]} | {year}")


before: 2019-01-01T00:00:00.000Z
after: 2019-02-01T00:00:00.000Z

before: 2019-02-01T00:00:00.000Z
after: 2019-03-01T00:00:00.000Z

before: 2019-03-01T00:00:00.000Z
after: 2019-04-01T00:00:00.000Z

before: 2019-04-01T00:00:00.000Z
after: 2019-05-01T00:00:00.000Z

before: 2019-05-01T00:00:00.000Z
after: 2019-06-01T00:00:00.000Z

before: 2019-06-01T00:00:00.000Z
after: 2019-07-01T00:00:00.000Z

before: 2019-07-01T00:00:00.000Z
after: 2019-08-01T00:00:00.000Z

before: 2019-08-01T00:00:00.000Z
after: 2019-09-01T00:00:00.000Z

before: 2019-09-01T00:00:00.000Z
after: 2019-10-01T00:00:00.000Z

before: 2019-10-01T00:00:00.000Z
after: 2019-11-01T00:00:00.000Z

before: 2019-11-01T00:00:00.000Z
after: 2019-12-01T00:00:00.000Z

before: 2019-12-01T00:00:00.000Z
after: 2020-01-01T00:00:00.000Z



In [ ]:
test_cases = [poly_1]

# safety check, only run cell if you specifically say "yes"
response = input("Are you sure you want to execute this cell? (yes/no): ")
if response.lower() == "yes":
    for i in range(len(year_iter) - 1):
        for test_case in test_cases:
            if test_case is None:
                continue

            i = 1  # iterator for names

            """
            Begin time count
            """
            t_before = time.time()

            """
            Define our filters
            """
            # define current geometry filter
            geometry_filter = {
                "type": "GeometryFilter",  # set filter type as geometry
                "field_name": "geometry",  # give json column containing geometry
                "config": test_case,  # input json
            }

            # DEMO DATE RANGE FILTER
            # ONLY FOR ONE MONTH
            date_range_filter = {
                "type": "DateRangeFilter",
                "field_name": "acquired",  # selects for when imagery was measured, not published
                "config": {
                    "gte": f"{year_iter[i]}-{month_nums[i]}-01T00:00:00.000Z",  # greater then equal to
                    "lt": f"{year_iter[i + 1]}-{month_nums[i + 1]}-01T00:00:00.000Z",  # less then
                },
            }

            # cloud filter
            cloud_cover_filter = {
                "type": "RangeFilter",  # generalized filter type that takes range of values
                "field_name": "cloud_cover",  # ask for cloud cover
                "config": {
                    "lt": 0.01  # filter for all scenes w <1% cloud cover
                },
            }

            # prevent restricted scenes from halting the request flow
            # permission_filter = {
            #     "type": "AssetFilter",
            #     "config": ["analytic_sr_udm2"]
            # }

            # permission_filter = {
            #     "type": "AssetFilter",
            #     "config": ["analytic_sr_udm2"]
            # }

            # combine filters
            combined_filters = {
                "type": "AndFilter",  # filter type for and conditional
                "config": [geometry_filter, date_range_filter, cloud_cover_filter],
            }

            """
            Take our filters and make a search request
            """
            # defines package type
            # PSScene has orthorectified 8 and 4 band imagery with the udm2 file
            # you can take imagery from multiple types but we only need this one
            item_types = ["PSScene"]

            # feed our filters and package type selection into a filter dict
            search_request = {"item_types": item_types, "filter": combined_filters}

            # and we search Planet labs imagery with it
            search_result = requests.post(  # post sends our request to https api
                "https://api.planet.com/data/v1/quick-search",
                auth=HTTPBasicAuth(planet_key, ""),  # we give it our key
                json=search_request,  # and our request dict
            )

            """
            Crunch all IDs into a list
            """
            ids = [feature["id"] for feature in search_result.json()["features"]]

            """
            Make our order request

            1. Setup our order json
            2. Setup our tools
            """
            # set content type to json
            headers = {"content-type": "application/json"}

            # init order parameters, this one has the 4 band
            product = [
                {
                    "item_ids": ids,
                    "item_type": "PSScene",
                    "product_bundle": "analytic_sr_udm2",  # ortho 4 band surface reflectance
                }
            ]

            order_request = {
                "name": "test",
                "products": product,
                "delivery": {
                    "single_archive": True,  # archive all bundles together in single file
                    "archive_type": "zip",  # get zip folder
                },
            }

            # init clip to sb county
            clip = {"clip": {"aoi": test_case}}

            # init ndvi calculation
            bandmath = {
                "bandmath": {
                    "b1": "b1",
                    "b2": "b2",
                    "b3": "b3",
                    "b4": "b4",
                    "b5": "(b4 - b3) / (b4 + b3)",
                }
            }

            # make name
            # order_name = f"{var_name_as_str(test_case)}_month_{num}_download_{i}"
            order_name = "error_test"

            # create request json
            tool_request = {
                "name": order_name,
                "products": product,
                "tools": [clip, bandmath],
                "delivery": {"single_archive": True, "archive_type": "zip"},
            }

            """
            Send in the order
            """
            # call the function

            retry_counter = 0

            order_url = place_order(tool_request, auth, retry_counter)

            """
            And test for order success DEPRECATED
            """
            # poll_for_success(orders_api_url, auth)

            """
            Wait 2 seconds before sending in next order
            """
            time.sleep(2)

            i = i + 1

            """
            End time count and output total time PARTIALLY DEPRECATED, MOVED TO OUTSIDE THE FOR LOOP
            """

    # now we can poll for success
    poll_for_success(orders_api_url, auth)

    t_after = time.time()
    locals()[f"{test_case}_exec_t"] = t_after - t_before

    print(
        f"Total execution time for {order_name}: {locals()[f'{test_case}_exec_t']} seconds"
    )

    # ________________End of code chunk___________________#

    print("Cell executed!")
else:
    print("Execution canceled.")

{'_links': {'_first': 'https://api.planet.com/data/v1/searches/41a4e1f56c1f4b798780fa143c93f62c/results?_page=eyJwYWdlX3NpemUiOiAyNTAsICJzb3J0X2J5IjogInB1Ymxpc2hlZCIsICJzb3J0X2Rlc2MiOiB0cnVlLCAic29ydF9zdGFydCI6IG51bGwsICJzb3J0X2xhc3RfaWQiOiBudWxsLCAic29ydF9wcmV2IjogZmFsc2UsICJxdWVyeV9wYXJhbXMiOiB7fX0%3D', '_next': 'https://api.planet.com/data/v1/searches/41a4e1f56c1f4b798780fa143c93f62c/results?_page=eyJwYWdlX3NpemUiOiAyNTAsICJzb3J0X2J5IjogInB1Ymxpc2hlZCIsICJzb3J0X2Rlc2MiOiB0cnVlLCAic29ydF9zdGFydCI6ICIyMDIxLTAzLTAyVDA5OjE5OjA3LjAwMDAwMFoiLCAic29ydF9sYXN0X2lkIjogIjIwMTkwMjA2XzE3NDIxMF8xXzEwNDMiLCAic29ydF9wcmV2IjogZmFsc2UsICJxdWVyeV9wYXJhbXMiOiB7fX0%3D', '_self': 'https://api.planet.com/data/v1/searches/41a4e1f56c1f4b798780fa143c93f62c/results?_page=eyJwYWdlX3NpemUiOiAyNTAsICJzb3J0X2J5IjogInB1Ymxpc2hlZCIsICJzb3J0X2Rlc2MiOiB0cnVlLCAic29ydF9zdGFydCI6IG51bGwsICJzb3J0X2xhc3RfaWQiOiBudWxsLCAic29ydF9wcmV2IjogZmFsc2UsICJxdWVyeV9wYXJhbXMiOiB7fX0%3D'}, 'features': [{'_links': {'_self': 'https://a

In [ ]:
# globals()["poly_1_exec_t"]

KeyError: 'poly_1_exec_t'

In [ ]:
# globals()[poly_10]

TypeError: unhashable type: 'dict'

In [ ]:
# # define filter for all scenes touching sb county
# geometry_filter = {
#     "type": "GeometryFilter", # set filter type as geometry
#     "field_name": "geometry", # give json column containing geometry
#     "config": poly_1 # input sb county json
# }


# # DEMO DATE RANGE FILTER
# # ONLY FOR ONE MONTH
# date_range_filter = {
#     "type": "DateRangeFilter",
#     "field_name": "acquired",  # selects for when imagery was measured, not published
#     "config": {
#         "gte": "2022-05-01T00:00:00.000Z", # greater then equal to
#         "lt":  "2022-06-01T00:00:00.000Z" # less then
#     }
# }


# # cloud filter
# cloud_cover_filter = {
#     "type": "RangeFilter", # generalized filter type that takes range of values
#     "field_name": "cloud_cover", # ask for cloud cover
#     "config": {
#         "lt": 0.01 # filter for all scenes w <1% cloud cover
#     }
# }

# # combine filters
# combined_filters = {
#     "type": "AndFilter", # filter type for and conditional
#     "config": [geometry_filter, date_range_filter, cloud_cover_filter]
# }



In [ ]:
# geometry_filter

{'type': 'GeometryFilter',
 'field_name': 'geometry',
 'config': {'type': 'MultiPolygon',
  'coordinates': [[[[-120.030715, 34.670007],
     [-120.030715, 34.669253],
     [-120.031252, 34.669253],
     [-120.031248, 34.669314],
     [-120.031489, 34.669834],
     [-120.031387, 34.670007],
     [-120.030715, 34.670007],
     [-120.030715, 34.670007]]]]}}

### Take our filters and make a search request

In [ ]:
# # defines package type
# # PSScene has orthorectified 8 and 4 band imagery with the udm2 file
# # you can take imagery from multiple types but we only need this one
# item_types = ["PSScene"]

# # feed our filters and package type selection into a filter dict
# search_request = {
#     "item_types": item_types,
#     "filter": combined_filters
# }

# # and we search Planet labs imagery with it
# search_result = \
#     requests.post( # post sends our request to https api
#         "https://api.planet.com/data/v1/quick-search",
#         auth = HTTPBasicAuth(planet_key, ''),  # we give it our key
#         json = search_request # and our request dict
#     )

# print(search_result)

<Response [200]>


In [ ]:
# ids = [feature['id'] for feature in search_result.json()["features"]]

# print(f"First 5 IDs...")
# for i in ids[0:5]:
#     print(f"\t{i}")

# print(f"\nTotal # of IDs: {len(ids)}")

First 5 IDs...
	20220524_181016_82_2231
	20220524_181014_61_2231
	20220530_182056_81_2495
	20220530_181932_52_2438
	20220529_174629_68_2429

Total # of IDs: 35


### Make our order request

In [ ]:
# # set content type to json
# headers = {"content-type": "application/json"}

# # init order parameters, this one has the 4 band
# product = [
#     {
#         "item_ids": ids,
#         "item_type": "PSScene",
#         "product_bundle": "analytic_sr_udm2", # ortho 4 band surface reflectance
#     }
# ]

# order_request = {
#     "name": "bandmath test",
#     "products": product,
#     "delivery": {
#         "single_archive": True, # archive all bundles together in single file
#         "archive_type": "zip" # get zip folder
#         }
# }

#### Setup tools

In [ ]:
# # init clip to sb county
# clip =  {
#     "clip": {
#         "aoi": poly_1
#     }
# }

# # init ndvi calculation
# bandmath = {
#     "bandmath": {
#         "b1": "b1",
#         "b2": "b2",
#         "b3": "b3",
#         "b4": "b4",
#         "b5": "(b4 - b3) / (b4 + b3)"
#     }
# }

# # create request json
# tool_request = {
#     "name": "bandmath_test",
#     "products": product,
#     "tools": [clip, bandmath],
#     "delivery": {"single_archive": True, "archive_type": "zip"}
# }

In [ ]:
# tool_request

{'name': 'tool_test',
 'products': [{'item_ids': ['20220524_181016_82_2231',
    '20220524_181014_61_2231',
    '20220530_182056_81_2495',
    '20220530_181932_52_2438',
    '20220529_174629_68_2429',
    '20220529_174627_39_2429',
    '20220526_180826_59_2233',
    '20220523_174801_90_241f',
    '20220523_174759_60_241f',
    '20220523_182151_31_2477',
    '20220520_181314_02_2231',
    '20220521_184036_62_2402',
    '20220522_182144_92_249d',
    '20220521_181911_68_2473',
    '20220519_174814_10_2465',
    '20220519_181826_70_248b',
    '20220517_174813_78_2453',
    '20220516_174828_98_245c',
    '20220516_183603_81_2402',
    '20220514_175142_02_2427',
    '20220512_174939_63_2420',
    '20220512_174941_93_2420',
    '20220512_174947_10_2465',
    '20220512_184124_16_2407',
    '20220511_174558_82_2447',
    '20220510_174914_10_2453',
    '20220506_182221_48_2484',
    '20220506_182136_31_248f',
    '20220504_183923_41_2403',
    '20220504_174556_17_2447',
    '20220504_174553_87_

### Make Order

##### Make Function

In [ ]:
# def place_order(request, auth):

#     # make order request
#     response = requests.post(
#         orders_api_url,
#         data = json.dumps(request),
#         auth = auth,
#         headers = headers
#         )
#     print(response.json())

#     # get ids of scenes

#     # print(response.json())
#     order_id = response.json()["id"]
#     print(order_id)

#     # construct the url of our order
#     order_url = orders_api_url + '/' + order_id

#     return order_url

##### send in the order

In [ ]:
# # safety check, only run cell if you specifically say "yes"
# response = input("Are you sure you want to execute this cell? (yes/no): ")
# if response.lower() == "yes":


#     # clip tool works!!
#     clip_url = place_order(tool_request, auth)


#     print("Cell executed!")
# else:
#     print("Execution canceled.")

{'_links': {'_self': 'https://api.planet.com/compute/ops/orders/v2/dbc5cddf-bae8-40c1-b8dd-16551c062d6a'}, 'created_on': '2025-04-30T05:17:56.532359Z', 'delivery': {'archive_filename': 'output.zip', 'archive_type': 'zip', 'single_archive': True}, 'error_hints': [], 'id': 'dbc5cddf-bae8-40c1-b8dd-16551c062d6a', 'last_message': 'Preparing order', 'last_modified': '2025-04-30T05:17:56.532359Z', 'name': 'tool_test', 'products': [{'item_ids': ['20220524_181016_82_2231', '20220524_181014_61_2231', '20220530_182056_81_2495', '20220530_181932_52_2438', '20220529_174629_68_2429', '20220529_174627_39_2429', '20220526_180826_59_2233', '20220523_174801_90_241f', '20220523_174759_60_241f', '20220523_182151_31_2477', '20220520_181314_02_2231', '20220521_184036_62_2402', '20220522_182144_92_249d', '20220521_181911_68_2473', '20220519_174814_10_2465', '20220519_181826_70_248b', '20220517_174813_78_2453', '20220516_174828_98_245c', '20220516_183603_81_2402', '20220514_175142_02_2427', '20220512_174939_

In [ ]:
# def poll_for_success(order_url, auth, num_loops = 60):
#     i = 0
#     while(i < num_loops):

#         # iterate
#         i += 1

#         # get order request
#         r = requests.get(order_url, auth = auth)
#         response = r.json()

#         # grab current state
#         state = response["orders"][0]["state"]
#         print(state)

#         # compare it to a variety of end states and print it
#         end_states = ["success", "failed", "partial"]
#         if state in end_states:
#             print(f"End State: {state}")
#             break

#         # wait 30 secs
#         time.sleep(30)

# poll_for_success(orders_api_url, auth)

success
End State: success


In [ ]:
test_case_sec = {
    "poly_1": 937.1823189258575,
    "poly_10": 1092.1013860702515,
    "poly_25": 1241.3868927955627,
    "poly_30": 636.5318565368652,
    "poly_40": 305.0745267868042,
    "vert_10": 636.7350268363953,
    "vert_25": 1122.617277622223,
    "vert_60": 484.223121881485,
    "vert_100": 363.7645287513733,
    "vert_150": 664.8928003311157,
    "vert_200": 665.0790801048279,
    "vert_300": 938.0900747776031,
    "vert_400": 1694.6172664165497,
}

In [ ]:
test_case_min = [time / 60 for time in list(test_case_sec.values())]
test_case_rems = [time % 60 for time in list(test_case_sec.values())]

for test_case, min, secs in zip(test_case_sec.keys(), test_case_min, test_case_rems):
    if secs < 10:
        secs = f"0{round(secs)}"
    else:
        secs = round(secs)
    print(f"{test_case} time: {round(min)}:{secs}")

poly_1 time: 16:37
poly_10 time: 18:12
poly_25 time: 21:41
poly_30 time: 11:37
poly_40 time: 5:05
vert_10 time: 11:37
vert_25 time: 19:43
vert_60 time: 8:04
vert_100 time: 6:04
vert_150 time: 11:05
vert_200 time: 11:05
vert_300 time: 16:38
vert_400 time: 28:15


### winner: Poly 40

In [ ]:
print(
    f"{list(test_case_sec.keys())[4]} time: {round(test_case_min[4])}:0{round(test_case_rems[4])}"
)

poly_40 time: 5:05
